# TITLE:🎬📽 Movies Revenue Prediction: From Data Exploration to Modeling
## Introduction

- This project analyzes **movie data sourced from The Movie Database (TMDB) API** to understand factors influencing movie revenue and build a model to predict it.
- The dataset includes films released *upto the year **2025***, containing details such as budget, revenue, popularity, vote average, genres, and release dates.

**Our goal**

- To explore the data, identify key patterns, and develop a predictive model capable of estimating a movie’s potential revenue based on available features.
  ## 1. 📁DATA COLLECTION 
- In this notebook we are going to fetch data using the TMDB(the movie database)API.
- We get the **movies summary using the (/movies/popular) endpoint**.
- Then we get **more details about the movies using the (/movies/movie_id endpoint)**.
- We **then merge** the two to have a dataset with the data we need for our project. 

In [1]:
#testing whether our api key works
from dotenv import load_dotenv
import os
import requests
load_dotenv(dotenv_path=r"C:\Users\Win\Desktop\API project\.env")
API_KEY=os.getenv('TMDB_API_KEY')
print(len(API_KEY))



32


In [2]:
#lets test whether our connection is working
#we define the base URL for our TMDB api
BASE_URL="https://api.themoviedb.org/3"

In [3]:
#creating a function to get a movie
def get_movie(movie_id):
    url=f"{BASE_URL}/movie/{movie_id}?api_key={API_KEY}&language=en_US"
    response= requests.get(url)
    if response.status_code==200: #if succesfull
        return response.json() #return the movie as a puthon dictionary
    else:
        return None

In [4]:
#Lets call the funtion
movie=get_movie(550)
print(movie)

{'adult': False, 'backdrop_path': '/5TiwfWEaPSwD20uwXjCTUqpQX70.jpg', 'belongs_to_collection': None, 'budget': 63000000, 'genres': [{'id': 18, 'name': 'Drama'}, {'id': 53, 'name': 'Thriller'}], 'homepage': 'http://www.foxmovies.com/movies/fight-club', 'id': 550, 'imdb_id': 'tt0137523', 'origin_country': ['US'], 'original_language': 'en', 'original_title': 'Fight Club', 'overview': 'A ticking-time-bomb insomniac and a slippery soap salesman channel primal male aggression into a shocking new form of therapy. Their concept catches on, with underground "fight clubs" forming in every town, until an eccentric gets in the way and ignites an out-of-control spiral toward oblivion.', 'popularity': 21.1346, 'poster_path': '/pB8BM7pdSp6B6Ih7QZ4DrQ3PmJK.jpg', 'production_companies': [{'id': 711, 'logo_path': '/tEiIH5QesdheJmDAqQwvtN60727.png', 'name': 'Fox 2000 Pictures', 'origin_country': 'US'}, {'id': 508, 'logo_path': '/4sGWXoboEkWPphI6es6rTmqkCBh.png', 'name': 'Regency Enterprises', 'origin_cou

In [5]:
#since the above data appearsto be messy, lets clean up
movie=get_movie(550)
print(f"title:{movie['title']}")
print(f"Release date:{movie['release_date']}")
print(f"Rating:{movie['vote_average']}")
print(f"Overview:{movie['overview']}")

title:Fight Club
Release date:1999-10-15
Rating:8.438
Overview:A ticking-time-bomb insomniac and a slippery soap salesman channel primal male aggression into a shocking new form of therapy. Their concept catches on, with underground "fight clubs" forming in every town, until an eccentric gets in the way and ignites an out-of-control spiral toward oblivion.


In [6]:
#now that we have confirmed that our connection is working , we go ahead and fetch foer more movies
import time
import pandas as pd
from tqdm import tqdm  # for progress bar 
#we define our function
def get_popular_movies(page=1):
    """Fetch one page (20 movies) of popular movies."""
    url = f"{BASE_URL}/movie/popular?api_key={API_KEY}&language=en-US&page={page}"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json().get('results', [])
    else:
        print(f"⚠️  Error {response.status_code} on page {page}")
        return []

# Fetch multiple pages safely
all_movies = []
num_pages = 200  # ~4000 movies

for page in tqdm(range(1, num_pages + 1), desc="Fetching popular movies"):
    movies = get_popular_movies(page)
    all_movies.extend(movies)
    time.sleep(0.25)  # pause to avoid rate limit



Fetching popular movies: 100%|███████████████████████████████████████████████████████| 200/200 [05:31<00:00,  1.66s/it]


In [10]:
# Convert to DataFrame and save
popular_df = pd.DataFrame(all_movies)
os.makedirs("data", exist_ok=True)
popular_df.to_csv("data/popular_movies.csv", index=False)

print(f"\n✅ Saved {len(popular_df)} movies to data/popular_movies.csv")
print(popular_df[['title', 'release_date', 'vote_average']].head())


✅ Saved 4000 movies to data/popular_movies.csv
                             title release_date  vote_average
0                        Our Fault   2025-10-15         7.797
1  Captain Hook - The Cursed Tides   2025-07-11         4.500
2                   Inside Furioza   2025-10-14         6.576
3                War of the Worlds   2025-07-29         4.300
4                      Stolen Girl   2025-09-04         6.686


In [12]:
popular_df.columns


Index(['adult', 'backdrop_path', 'genre_ids', 'id', 'original_language',
       'original_title', 'overview', 'popularity', 'poster_path',
       'release_date', 'title', 'video', 'vote_average', 'vote_count'],
      dtype='object')

In [13]:
# --- Function to fetch full details for one movie ---
def get_movie_details(movie_id):
    url = f"{BASE_URL}/movie/{movie_id}?api_key={API_KEY}&language=en-US"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        return None

# --- Fetch details for all movies ---
details_list = []
for mid in tqdm(popular_df['id'], desc="Fetching movie details"):
    data = get_movie_details(mid)
    if data:
        details_list.append({
            'id': data.get('id'),
            'title': data.get('title'),
            'budget': data.get('budget'),
            'revenue': data.get('revenue'),
            'runtime': data.get('runtime'),
            'status': data.get('status'),
            'release_date': data.get('release_date'),
            'popularity': data.get('popularity'),
            'vote_average': data.get('vote_average'),
            'vote_count': data.get('vote_count'),
            'genres': [g['name'] for g in data.get('genres', [])]
        })
    time.sleep(0.25)  # avoid rate limits

details_df = pd.DataFrame(details_list)

# --- Merge the details with your original summary dataset ---
merged_df = pd.merge(popular_df, details_df, on='id', how='left')

# --- Save the merged file ---
os.makedirs("data", exist_ok=True)
merged_df.to_csv("data/movies_full.csv", index=False)

print(f"\n✅ Merged dataset saved: data/movies_full.csv")
print(merged_df[['title_x', 'budget', 'revenue', 'runtime']].head())

Fetching movie details: 100%|████████████████████████████████████████████████████| 4000/4000 [1:55:18<00:00,  1.73s/it]



✅ Merged dataset saved: data/movies_full.csv
                           title_x    budget  revenue  runtime
0                        Our Fault         0        0      113
1  Captain Hook - The Cursed Tides         0        0       90
2                   Inside Furioza         0        0      166
3                War of the Worlds         0        0       91
4                      Stolen Girl  26000000    92691      110


In [14]:
merged_df.columns


Index(['adult', 'backdrop_path', 'genre_ids', 'id', 'original_language',
       'original_title', 'overview', 'popularity_x', 'poster_path',
       'release_date_x', 'title_x', 'video', 'vote_average_x', 'vote_count_x',
       'title_y', 'budget', 'revenue', 'runtime', 'status', 'release_date_y',
       'popularity_y', 'vote_average_y', 'vote_count_y', 'genres'],
      dtype='object')

## NEXT WE MOVE TO DATA CLEANING